# 🤖 Kaizen OS — SFT + GRPO Training Notebook

**End-to-end re-runnable training script for the Kaizen OS agent.**

This notebook:
1. Installs all dependencies (Unsloth, TRL, etc.)
2. Clones the kaizen2 repo from GitHub
3. Runs **Supervised Fine-Tuning (SFT)** on Qwen2.5-3B-Instruct with LoRA
4. Runs **GRPO Reinforcement Learning** from the SFT checkpoint
5. Pushes both adapters to HuggingFace Hub
6. Displays loss + reward curves inline

**Runtime:** GPU (T4 minimum, A100 recommended)  
**Est. time:** ~60–90 min on T4, ~30–40 min on A100  

---
**Links:**
- HF Space: https://huggingface.co/spaces/NehaChikle/kaizen-os
- SFT Model: https://huggingface.co/NehaChikle/kaizen-qwen2.5-3b-sft
- GRPO Model: https://huggingface.co/NehaChikle/kaizen-qwen2.5-3b-grpo
- GitHub: https://github.com/ChikleNeha/kaizen2

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('No GPU detected! Go to Runtime → Change runtime type → GPU')

In [ ]:
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Install Dependencies

In [ ]:
# Unsloth — fastest fine-tuning library with 4-bit support
# Uses colab-new variant which auto-detects CUDA version
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# TRL >= 0.13 required for stable GRPOConfig with save_steps support
!pip install -q \
    "trl>=0.13.0" \
    "datasets>=2.18.0" \
    "accelerate>=0.28.0" \
    "bitsandbytes>=0.43.0" \
    "peft>=0.10.0" \
    "transformers>=4.40.0" \
    "matplotlib" \
    "huggingface_hub>=1.0.0" \
    "pydantic>=2.0.0" \
    "psutil>=5.9.0" \
    "gymnasium>=0.29.0"

print('✅ All dependencies installed.')

## 2. Clone the Kaizen2 Repo

In [ ]:
import os

GITHUB_REPO = 'https://github.com/ChikleNeha/kaizen2'
WORKDIR     = '/content/kaizen2'

if os.path.isdir(WORKDIR):
    print(f'Repo already cloned at {WORKDIR}, pulling latest...')
    !cd {WORKDIR} && git pull
else:
    !git clone --depth=1 {GITHUB_REPO} {WORKDIR}

os.chdir(WORKDIR)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# Verify all required files exist
import json

required = [
    'training/sft_train.py',
    'training/grpo_train.py',
    'training/golden_examples.json',
]
for path in required:
    assert os.path.exists(path), f'MISSING: {path}'
    print(f'✅ {path}')

with open('training/golden_examples.json') as f:
    examples = json.load(f)
print(f'\nTraining examples: {len(examples)}')

## 3. HuggingFace Login

You need a HF token with **write** access.  
Get one at: https://huggingface.co/settings/tokens

In Colab: go to **Secrets** (🔑 icon in the left sidebar) and add `HF_TOKEN`.

In [ ]:
from huggingface_hub import login

# Try Colab secrets first, then fall back to manual input
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab secrets.')
except Exception:
    import getpass
    hf_token = getpass.getpass('Enter your HuggingFace token (write access): ')

os.environ['HF_TOKEN']   = hf_token
os.environ['HF_REPO_ID'] = 'NehaChikle/kaizen-qwen2.5-3b-sft'

login(token=hf_token, add_to_git_credential=False)
print('✅ HuggingFace login OK')

## 4. Phase 1 — Supervised Fine-Tuning (SFT)

Fine-tunes **Qwen2.5-3B-Instruct** with LoRA (rank 16) on the golden examples dataset.  
Uses Alpaca-style instruction format. Saves adapter to `/content/kaizen_sft_model`.

In [ ]:
import sys
sys.path.insert(0, WORKDIR)

os.environ['SFT_OUTPUT_DIR'] = '/content/kaizen_sft_model'
os.environ['HF_REPO_ID']     = 'NehaChikle/kaizen-qwen2.5-3b-sft'

print('Starting SFT training...')
print('=' * 60)

In [ ]:
# Run SFT training inline (imports from cloned repo)
# This cell takes ~25-40 min on T4

import importlib.util, types

spec = importlib.util.spec_from_file_location('sft_train', f'{WORKDIR}/training/sft_train.py')
sft_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sft_module)

sft_output_path = sft_module.train()
print(f'\n✅ SFT complete. Adapter saved to: {sft_output_path}')

In [ ]:
# Display SFT loss curve
from IPython.display import Image, display
import os

loss_curve_path = os.path.join(WORKDIR, 'loss_curve.png')
if os.path.exists(loss_curve_path):
    print('SFT Loss Curve:')
    display(Image(loss_curve_path))
else:
    # Try fallback location
    loss_curve_path = './loss_curve.png'
    if os.path.exists(loss_curve_path):
        display(Image(loss_curve_path))
    else:
        print('loss_curve.png not found — check sft_train.py output.')

In [ ]:
# Verify SFT adapter files
sft_dir = '/content/kaizen_sft_model'
assert os.path.isdir(sft_dir), f'SFT output dir missing: {sft_dir}'
files = os.listdir(sft_dir)
print(f'SFT adapter files ({len(files)}):')
for f in sorted(files):
    size = os.path.getsize(os.path.join(sft_dir, f))
    print(f'  {f:40s}  {size/1e6:.1f} MB')

## 5. Phase 2 — GRPO Reinforcement Learning

Loads the SFT checkpoint and further trains it with **Group Relative Policy Optimisation (GRPO)**.  
The reward function uses the `KaizenEnv` environment — chaos resolution gives the highest reward (+3.0 bonus).  
Saves final policy to `/content/kaizen_grpo_model`.

In [ ]:
os.environ['KAIZEN_SFT_PATH'] = '/content/kaizen_sft_model'
os.environ['GRPO_OUTPUT_DIR'] = '/content/kaizen_grpo_model'
# HF_REPO_ID stays as the SFT repo; grpo_train.py will NOT push (shell script owns that)

print('Starting GRPO training...')
print('Reward range: -10.0 (protected kill) → ~+8.0 (full chaos resolution)')
print('=' * 60)

In [ ]:
# Run GRPO training inline
# This cell takes ~35-50 min on T4

spec = importlib.util.spec_from_file_location('grpo_train', f'{WORKDIR}/training/grpo_train.py')
grpo_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grpo_module)

grpo_output_path = grpo_module.train()
print(f'\n✅ GRPO complete. Final model at: {grpo_output_path}')

In [ ]:
# Display GRPO reward curve
reward_curve_path = os.path.join(WORKDIR, 'reward_curve.png')
if os.path.exists(reward_curve_path):
    print('GRPO Reward Curve:')
    display(Image(reward_curve_path))
else:
    reward_curve_path = './reward_curve.png'
    if os.path.exists(reward_curve_path):
        display(Image(reward_curve_path))
    else:
        print('reward_curve.png not found — check grpo_train.py output.')

## 6. Push Both Adapters to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

api      = HfApi()
token    = os.environ['HF_TOKEN']
sft_repo = 'NehaChikle/kaizen-qwen2.5-3b-sft'
grpo_repo = 'NehaChikle/kaizen-qwen2.5-3b-grpo'

# Push SFT adapter
sft_dir = '/content/kaizen_sft_model'
if os.path.isdir(sft_dir) and os.listdir(sft_dir):
    print(f'Pushing SFT adapter → {sft_repo} ...')
    api.create_repo(sft_repo, repo_type='model', exist_ok=True, token=token)
    api.upload_folder(
        folder_path=sft_dir,
        repo_id=sft_repo,
        repo_type='model',
        token=token,
        commit_message='Kaizen OS SFT adapter — LoRA Qwen2.5-3B',
    )
    print(f'✅ SFT → https://huggingface.co/{sft_repo}')
else:
    print('⚠️  SFT dir empty — skipping.')

# Push GRPO adapter
grpo_dir = '/content/kaizen_grpo_model'
if os.path.isdir(grpo_dir) and os.listdir(grpo_dir):
    print(f'Pushing GRPO adapter → {grpo_repo} ...')
    api.create_repo(grpo_repo, repo_type='model', exist_ok=True, token=token)
    api.upload_folder(
        folder_path=grpo_dir,
        repo_id=grpo_repo,
        repo_type='model',
        token=token,
        commit_message='Kaizen OS GRPO adapter — RL fine-tuned from SFT',
    )
    print(f'✅ GRPO → https://huggingface.co/{grpo_repo}')
else:
    print('⚠️  GRPO dir empty — skipping.')

## 7. Commit Curves to GitHub Repo

The judges require `loss_curve.png` and `reward_curve.png` committed as image files in the repo.

In [ ]:
import shutil

# Copy curves into the repo folder so they can be committed
for fname in ['loss_curve.png', 'reward_curve.png']:
    src_candidates = [
        f'/content/kaizen2/{fname}',
        f'/content/{fname}',
        f'./{fname}',
    ]
    for src in src_candidates:
        if os.path.exists(src):
            dst = os.path.join(WORKDIR, fname)
            shutil.copy2(src, dst)
            print(f'✅ Copied {fname} → {dst}')
            break
    else:
        print(f'⚠️  {fname} not found in any expected location.')

In [ ]:
# Configure git and push curves to GitHub
# NOTE: You need a GitHub Personal Access Token with repo write access
# for this cell. Skip it and push manually if you prefer.

try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
except Exception:
    import getpass
    gh_token = getpass.getpass('GitHub PAT (repo write access) — or press Enter to skip: ')

if gh_token and gh_token.strip():
    repo_url = f'https://{gh_token}@github.com/ChikleNeha/kaizen2.git'
    !cd {WORKDIR} && git config user.email 'kaizen-bot@training.run'
    !cd {WORKDIR} && git config user.name 'Kaizen Training Bot'
    !cd {WORKDIR} && git add loss_curve.png reward_curve.png
    !cd {WORKDIR} && git diff --cached --quiet || git commit -m 'Add training curves [auto]'
    !cd {WORKDIR} && git push {repo_url} HEAD:main
    print('✅ Curves committed and pushed to GitHub.')
else:
    print('Skipped GitHub push. Please commit loss_curve.png and reward_curve.png manually.')

## 8. Final Summary

In [ ]:
from IPython.display import display, Image
import os

print('=' * 60)
print('  KAIZEN OS — Training Complete')
print('=' * 60)
print()
print('Models pushed to HuggingFace Hub:')
print('  SFT  → https://huggingface.co/NehaChikle/kaizen-qwen2.5-3b-sft')
print('  GRPO → https://huggingface.co/NehaChikle/kaizen-qwen2.5-3b-grpo')
print()
print('Next steps:')
print('  1. In your HF Space (NehaChikle/kaizen-os), set:')
print('       KAIZEN_MODEL_NAME = NehaChikle/kaizen-qwen2.5-3b-sft')
print('       KAIZEN_DEMO_MODE  = false')
print('  2. Make sure loss_curve.png and reward_curve.png are in your GitHub repo')
print('  3. Link this notebook in your README')
print()

# Show both curves side by side
for label, path_candidates in [
    ('SFT Loss Curve', [f'{WORKDIR}/loss_curve.png', './loss_curve.png']),
    ('GRPO Reward Curve', [f'{WORKDIR}/reward_curve.png', './reward_curve.png']),
]:
    for p in path_candidates:
        if os.path.exists(p):
            print(f'{label}:')
            display(Image(p))
            break